# Model Uncertainty in Climate Projections

This notebook explores CMIP6 multi-model spread for **California** under **SSP3-7.0**, using data retrieved via the Cal-Adapt Analytics Engine. It builds a progression of visualizations that characterize *how much* models disagree, *where* they disagree, and *which variables* carry the most uncertainty.

**Workflow:**
1. Retrieve and preprocess CMIP6 multi-model temperature data
2. Compute annual anomalies relative to the 1850–1900 pre-industrial baseline
3. Visualize model spread: plume chart → ridgeline distributions → spatial diagnostics → warming-level scatter
4. Compare inter-model spread across variables (temperature vs. precipitation)

## Step 0: Setup — imports and configuration

In [ ]:
import warnings

import xarray as xr
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.lines as mlines
import holoviews as hv
import hvplot.xarray
import ridgeplot
import plotly.graph_objects as go
import ipywidgets as widgets
from ipywidgets import interact
import os

from bokeh.models import HoverTool
from bokeh.io import export_png
from bokeh.embed import file_html
from bokeh.resources import INLINE
from bokeh.models import HoverTool, ColumnDataSource, Div
from bokeh.layouts import column

from climakitae.explore.uncertainty import (
    CmipOpt,
    grab_multimodel_data,
    weighted_temporal_mean,
    calc_anom,
    cmip_mmm,
)
from climakitae.core.paths import GWL_1850_1900_FILE
from climakitae.util.utils import read_csv_file
from climakitae.util.colormap import read_ae_colormap

warnings.filterwarnings("ignore")
hv.extension("bokeh")
hv.plotting.bokeh.element.ElementPlot.active_tools = ['pan']
%config InlineBackend.figure_format = 'retina'

# ── Mode bar: no Plotly logo, no zoom/pan tools — download PNG only ────────
config = dict(
    displaylogo=False,
    scrollZoom=False,
    modeBarButtonsToRemove=[
        "zoom2d",
        "pan2d",
        "select2d",
        "lasso2d",
        "zoomIn2d",
        "zoomOut2d",
        "autoScale2d",
        "resetScale2d",
    ],
    # keep only "toImage" (download as png) visible on the mode bar
)

In [ ]:
os.makedirs("figures", exist_ok=True)
os.makedirs("figures/static", exist_ok=True)
os.makedirs("figures/html", exist_ok=True)

### Step 1: Retrieve CMIP6 multi-model data

We retrieve monthly near-surface air temperature (`tas`) for all available CMIP6 models with both historical and SSP3-7.0 simulations, subsetted to California.

In [ ]:
# Configure data options
copt = CmipOpt()
copt.variable = "tas"  # near-surface air temperature
copt.area_subset = "states"
copt.location = "California"
copt.area_average = False  # retain spatial dimensions
copt.timescale = "Amon"  # monthly

In [ ]:
# Retrieve and pre-process all available CMIP6 models
mdls_ds = grab_multimodel_data(copt)
mdls_ds

### Step 2: Compute annual anomalies and multi-model mean

We calculate annual averages, convert to temperature anomalies relative to a 1850–1900 pre-industrial baseline, and compute the multi-model mean (MMM). The anomaly framing places results in a [global warming levels](warming_levels.ipynb) context.

In [ ]:
# Annual average → Celsius → anomaly relative to 1850–1900 → multi-model mean
xy_ds_yr = weighted_temporal_mean(mdls_ds).compute()
xy_ds_yr = xy_ds_yr - 273.15
xy_ds_yr.tas.attrs["units"] = "°C"

xy_anom = calc_anom(xy_ds_yr, base_start=1850, base_end=1900)
xy_anom.tas.attrs["units"] = "°C"

xy_anom_mmm = cmip_mmm(xy_anom)

In [ ]:
# Area-average timeseries; split into historical and future periods
hist_start, hist_end, ssp_end = 1950, 2014, 2100

cmip_anom = xy_anom.groupby("time").mean(dim=["x", "y"])
cmip_anom.tas.attrs["units"] = "°C"
cmip_anom_mmm = cmip_mmm(cmip_anom)

hist_anom = cmip_anom.sel(time=slice(hist_start, hist_end))
hist_anom_mmm = cmip_anom_mmm.sel(time=slice(hist_start, hist_end))
ssp_anom = cmip_anom.sel(time=slice(hist_end, ssp_end))
ssp_anom_mmm = cmip_anom_mmm.sel(time=slice(hist_end, ssp_end))

In [ ]:
# Cal-Adapt: Analytics Engine model subset
cae_mdls_ls = [
    "FGOALS-g3",
    "EC-Earth3-Veg",
    "CESM2",
    "CNRM-ESM2-1",
    "MIROC6",
    "MPI-ESM1-2-HR",
    "EC-Earth3",
    "TaiESM1",
]
cmip_mdls_ls = [
    cmip_anom.simulation.values[i]
    for i in range(len(cmip_anom.simulation.values))
    if cmip_anom.simulation.values[i] not in cae_mdls_ls
]

cae_anom = cmip_anom.sel(simulation=cae_mdls_ls,
                         time=slice(hist_start, ssp_end))

### Step 3: Visualize the model spread

#### Step 3a: Select a warming level of interest

We use the year each model reaches a global warming level to anchor cross-model comparisons on physically comparable climate states, rather than a fixed calendar year.

In [ ]:
# Set warming level — options: 1.5, 2.0, 3.0, 4.0
warm_level = 3.0

In [ ]:
# Identify the year each model reaches warm_level
gwl_times = read_csv_file(GWL_1850_1900_FILE, index_col=[0, 1, 2])

scenario = "ssp370"
sim_idx = []
for simulation in ssp_anom.simulation.values:
    if simulation in gwl_times.index:
        if simulation == "CESM2":
            sim_idx.append((simulation, "r11i1p1f1", scenario))
        elif simulation == "CNRM-ESM2-1":
            sim_idx.append((simulation, "r1i1p1f2", scenario))
        else:
            sim_idx.append((simulation, "r1i1p1f1", scenario))

xy_da_list = []
year_reached_by_sim = []
for i in sim_idx:
    year_str = str(gwl_times[str(warm_level)].loc[i])[:4]
    if len(year_str) != 4:
        print(f"{warm_level}°C warming level not reached for {i[0]}")
    else:
        year_reached_by_sim.append((i, int(year_str)))
        xy_da_list.append(xy_anom.sel(time=int(year_str), simulation=i[0]))

thresh_df = pd.DataFrame(
    data=year_reached_by_sim, columns=[
        "simulation", "year_warming_level_reached"]
)
xy_by_warmlevel = xr.concat(xy_da_list, dim="simulation")

#### Step 3b: Plume chart — full model spread through 2100

A plume chart improves on a spaghetti plot by foregrounding the **10th–90th percentile envelope** (practical planning range), the **multi-model mean** (central estimate), and the **Cal-Adapt AE subset** highlighted within the broader CMIP6 archive.

In [ ]:
# cmip_anom = cmip_anom.sel(time=cmip_anom.time.isin(range(1980, 2100)))
# hist_anom = hist_anom.sel(time=hist_anom.time.isin(range(1980, 2100)))

full = (
    xy_anom.sel(time=xy_anom.time.isin(range(1980, 2100)))
    .groupby("time")
    .mean(dim=["x", "y"])
)
full.tas.attrs["units"] = "°C"
full_mmm = cmip_mmm(full)

In [ ]:
# Stack all model timeseries into (n_models, n_years) for percentile calculation
all_sims = full.simulation.values
all_years = full.time.values

sim_array = np.array([full.sel(simulation=s).tas.values for s in all_sims])

p1 = np.nanpercentile(sim_array, 1, axis=0)
p2 = np.nanpercentile(sim_array, 2, axis=0)
p10 = np.nanpercentile(sim_array, 10, axis=0)
p25 = np.nanpercentile(sim_array, 25, axis=0)
p75 = np.nanpercentile(sim_array, 75, axis=0)
p90 = np.nanpercentile(sim_array, 90, axis=0)
p98 = np.nanpercentile(sim_array, 98, axis=0)
p99 = np.nanpercentile(sim_array, 99, axis=0)
mmm = np.nanmean(sim_array, axis=0)

cae_indices = [i for i, s in enumerate(all_sims) if s in cae_mdls_ls]

In [ ]:
def remove_bokeh_logo(plot, element):
    plot.state.toolbar.logo = None

In [ ]:
cmip_anom = full.sel(simulation=~full.simulation.isin(
    cae_anom.simulation.values))
cae_anom = full.sel(simulation=full.simulation.isin(
    cae_anom.simulation.values))

In [ ]:
"""
Pure-Plotly rewrite of the CMIP6 / Cal-Adapt AE temperature projection plot.

Assumes the data-prep block you shared has already run, so the following
variables exist in scope:

    cmip_anom, cae_anom   xarray Datasets, dims (time, simulation), var "tas"
    full_mmm              xarray Dataset/DataArray, dim (time,), var "tas"
    all_years, p2, p98    1-D numpy arrays (from the percentile calc)
    thresh_df             pandas DataFrame with a "year_warming_level_reached" column
    hist_end              int/float, year the historical period ends
    warm_level             the warming level (e.g. 2.0) used in the legend label

No HoloViews / Bokeh anywhere below — just plotly.graph_objects.
"""

import os
import numpy as np
import plotly.graph_objects as go

# ── Colors ──────────────────────────────────────────────────────────────────
SSP_COLOR = "#2166AC"  # strong mid-blue — CMIP6 (many lines)
CAE_COLOR = "#D6604D"  # muted red-orange — Cal-Adapt (fewer lines)
LW = 0.8
ALPHA = 0.30

fig = go.Figure()

# ── 2nd–98th percentile band ────────────────────────────────────────────────
# lower bound (invisible), then upper bound filled down to it
fig.add_trace(
    go.Scatter(
        x=all_years,
        y=p2,
        mode="lines",
        line=dict(width=0),
        hoverinfo="skip",
        showlegend=False,
    )
)
fig.add_trace(
    go.Scatter(
        x=all_years,
        y=p98,
        mode="lines",
        line=dict(width=0),
        fill="tonexty",
        fillcolor="rgba(232, 149, 109, 0.45)",  # #E8956D @ 0.45 alpha
        hoverinfo="skip",
        showlegend=True,
        name="2nd-98th percentile",
    )
)

# ── Warming-level reached band (vrect) ──────────────────────────────────────
wl_min = int(thresh_df["year_warming_level_reached"].min())
wl_max = int(thresh_df["year_warming_level_reached"].max())
fig.add_vrect(
    x0=wl_min,
    x1=wl_max,
    fillcolor="#333333",
    opacity=0.07,
    line_width=0,
    layer="below",
)

# ── Individual CMIP6 model lines ────────────────────────────────────────────
for sim in cmip_anom.simulation.values:
    da = cmip_anom.sel(simulation=sim)
    fig.add_trace(
        go.Scatter(
            x=da.time.values,
            y=da.tas.values,
            mode="lines",
            line=dict(color=SSP_COLOR, width=LW),
            opacity=ALPHA,
            name=str(sim),
            legendgroup="cmip6",
            showlegend=False,
            hovertemplate="%{fullData.name}<br>Year: %{x}<br>%{y:.2f} \u00b0C<extra></extra>",
        )
    )

# ── Individual Cal-Adapt AE model lines ─────────────────────────────────────
for sim in cae_anom.simulation.values:
    da = cae_anom.sel(simulation=sim)
    fig.add_trace(
        go.Scatter(
            x=da.time.values,
            y=da.tas.values,
            mode="lines",
            line=dict(color=CAE_COLOR, width=LW),
            opacity=ALPHA,
            name=str(sim),
            legendgroup="cae",
            showlegend=False,
            hovertemplate="%{fullData.name}<br>Year: %{x}<br>%{y:.2f} \u00b0C<extra></extra>",
        )
    )

# ── Multi-model mean ────────────────────────────────────────────────────────
fig.add_trace(
    go.Scatter(
        x=full_mmm.time.values,
        y=full_mmm.tas.values,
        mode="lines",
        line=dict(color="#1a1a1a", width=LW * 3),
        name="Multi-model mean",
        hovertemplate="Multi-model mean<br>Year: %{x}<br>%{y:.2f} \u00b0C<extra></extra>",
    )
)

# ── Reference lines ──────────────────────────────────────────────────────────
fig.add_vline(
    x=hist_end,
    line=dict(color="#555555", width=0.8, dash="dash"),
    opacity=0.6,
)
fig.add_hline(
    y=0,
    line=dict(color="#999999", width=0.6, dash="dot"),
)

# ── Legend-only entries (mirrors the nan-curve trick from HoloViews) ───────
fig.add_trace(
    go.Scatter(
        x=[None],
        y=[None],
        mode="lines",
        line=dict(color=SSP_COLOR, width=LW * 2),
        name="CMIP6 models",
    )
)
fig.add_trace(
    go.Scatter(
        x=[None],
        y=[None],
        mode="lines",
        line=dict(color=CAE_COLOR, width=LW * 2),
        name="Cal-Adapt AE models",
    )
)
fig.add_trace(
    go.Scatter(
        x=[None],
        y=[None],
        mode="lines",
        line=dict(color="#333333", width=8),
        opacity=0.25,
        name=f"Years models reaching {warm_level} C",
    )
)
fig.add_trace(
    go.Scatter(
        x=[None],
        y=[None],
        mode="lines",
        line=dict(color="#555555", width=1.5, dash="dash"),
        opacity=0.6,
        name=f"SSP3-7.0 start ({hist_end})",
    )
)

# ── Y-axis range (padded around the full model spread) ─────────────────────
y_all = np.concatenate(
    [cmip_anom.tas.values.ravel(), cae_anom.tas.values.ravel()]
)
y_all = y_all[~np.isnan(y_all)]
y_min, y_max = y_all.min() - 0.25, y_all.max() + 0.25

# ── Layout ────────────────────────────────────────────────────────────────
fig.update_layout(
    title="CMIP6 California surface temperature projections (SSP3-7.0)",
    xaxis=dict(
        title="Year", range=[1980, 2100], showgrid=False, fixedrange=True
    ),
    yaxis=dict(
        title="Temperature anomaly (C) relative to 1850-1900",
        range=[y_min, y_max],
        showgrid=False,
        fixedrange=True,
    ),
    width=900,
    height=420,
    legend=dict(x=0.01, y=0.99, xanchor="left", yanchor="top"),
    hovermode="closest",
    dragmode=False,  # no box/lasso zoom-drag
    plot_bgcolor="white",
    paper_bgcolor="white",
    margin=dict(l=60, r=20, t=50, b=50),
)

# ── Mode bar: no Plotly logo, no zoom/pan tools — download PNG only ────────
config = dict(
    displaylogo=False,
    scrollZoom=False,
    modeBarButtonsToRemove=[
        "zoom2d",
        "pan2d",
        "select2d",
        "lasso2d",
        "zoomIn2d",
        "zoomOut2d",
        "autoScale2d",
        "resetScale2d",
    ],
    # keep only "toImage" (download as png) visible on the mode bar
)

fig.show(config=config)

# ── Export ───────────────────────────────────────────────────────────────
os.makedirs("figures/static", exist_ok=True)
os.makedirs("figures/html", exist_ok=True)

fig.write_image("figures/static/temperature_projections.png",
                scale=2.0)  # needs kaleido
fig.write_html(
    "figures/html/temperature_projections.html",
    include_plotlyjs="cdn",
    full_html=False,
    config=config,
)

#### Step 3c: Ridgeline plot — per-model distributions for a chosen period

Each ridge shows one model's distribution of annual temperature anomalies. Wide, flat ridges indicate high within-model spread; non-overlapping ridges indicate high between-model disagreement. Modify `ridge_start` / `ridge_end` to compare near-term vs. end-of-century.

In [ ]:
# Period for ridgeline — modify to compare near-term vs. end-of-century
ridge_start, ridge_end = 2071, 2100  # end-of-century

##### Approach A — annual anomalies (pre-computed `cae_anom` / `cmip_anom`)

Uses the annually-averaged, area-averaged anomalies already computed in Step 2. Quick to run but limited to annual resolution.

In [ ]:
ridge_start, ridge_end = 2071, 2100

#  separate the two groups before building traces
cae_data = cae_anom.sel(time=slice(ridge_start, ridge_end))
cmip_data = cmip_anom.sel(time=slice(ridge_start, ridge_end))


def build_samples(da):
    samples, labels = [], []
    for m in da.simulation.values:
        vals = da.sel(simulation=m).tas.values
        vals = vals[~np.isnan(vals)]
        if len(vals) > 0:
            samples.append([vals])
            labels.append(m)
    return samples, labels


cae_samples, cae_labels = build_samples(cae_data)
cmip_samples, cmip_labels = build_samples(cmip_data)

cmap = read_ae_colormap(cmap="ae_diverging", cmap_hex=True)
cmap = "plasma"

#  build one figure per group


def make_ridge_fig(samples, labels):
    return ridgeplot.ridgeplot(
        samples=samples,
        labels=labels,
        colorscale=cmap,
        opacity=0.75,
        bandwidth=0.4,
        kde_points=1000,
        spacing=0.25,
        line_width=1.5,
        xpad=1.0,
    )


fig_cae = make_ridge_fig(cae_samples, cae_labels)
fig_cmip = make_ridge_fig(cmip_samples, cmip_labels)

n_cae = len(fig_cae.data)
n_cmip = len(fig_cmip.data)

#  get full x range across BOTH figures
all_x = []
for trace in list(fig_cae.data) + list(fig_cmip.data):
    if trace.x is not None:
        all_x.extend([v for v in trace.x if v is not None and not np.isnan(v)])

x_min = min(all_x) - 0.3
x_max = max(all_x) + 0.3

#  combine into one figure — CMIP first, then CAE
fig = go.Figure()

for trace in fig_cmip.data:
    trace.visible = True
    fig.add_trace(trace)

for trace in fig_cae.data:
    trace.visible = False
    fig.add_trace(trace)

#  dropdown visibility masks
show_cmip = [True] * n_cmip + [False] * n_cae
show_cae = [False] * n_cmip + [True] * n_cae

#  layout helpers


def fig_height(labels, factor=25):
    return factor * len(labels) + 100


def fig_width(labels, factor=25):
    return 600


MARGIN = dict(l=60, r=30, t=100, b=60)

fig.update_layout(
    title=dict(
        text=f"All CMIP6 models — {ridge_start}–{ridge_end} (SSP3-7.0)",
        x=0.5,
        xanchor="center",
        y=0.98,
        yanchor="top",
    ),
    xaxis_title="Temperature anomaly (°C) relative to 1850–1900",
    height=fig_height(cmip_labels, factor=25),
    width=fig_width(cmip_labels, factor=25),
    showlegend=False,
    plot_bgcolor="white",
    margin=MARGIN,
    updatemenus=[
        dict(
            buttons=[
                dict(
                    label="All CMIP6 models",
                    method="update",
                    args=[
                        {"visible": show_cmip},
                        {
                            "title.text": f"All CMIP6 models — {ridge_start}–{ridge_end} (SSP3-7.0)",
                            "height": fig_height(cmip_labels, factor=25),
                            "width": fig_width(cmip_labels, factor=25),
                            "margin": MARGIN,
                        },
                    ],
                ),
                dict(
                    label="Cal-Adapt AE models",
                    method="update",
                    args=[
                        {"visible": show_cae},
                        {
                            "title.text": f"Cal-Adapt AE models — {ridge_start}–{ridge_end} (SSP3-7.0)",
                            "height": fig_height(cae_labels, factor=50),
                            "width": fig_width(cae_labels, factor=50),
                            "margin": MARGIN,
                        },
                    ],
                ),
            ],
            direction="down",
            showactive=True,
            x=0.0,
            xanchor="left",
            y=1.08,
            yanchor="top",
        )
    ],
)

#  axes
fig.update_xaxes(
    range=[x_min, x_max],
    title_text="Temperature anomaly (°C) relative to 1850–1900",
)

fig.update_yaxes(
    showticklabels=False,
    title_text="Model",
)

fig.show(config=config)
fig.write_html(
    "figures/html/ridgeplot.html",
    include_plotlyjs="cdn",
    full_html=False,
    config=config,
)
fig.write_image("figures/static/ridgeplot.png", scale=3)

##### Approach B — monthly anomalies (raw `mdls_ds`, custom baseline)

Reloads raw monthly data and subtracts a 1971–2010 reference climatology. Provides finer within-period spread at the cost of a longer compute step (uses Dask).

Dashed lines mark each model's **median**. Switch to the near-term period `(2021, 2050)` and rerun — the ridges converge substantially, reflecting lower model uncertainty at shorter time horizons.

#### Step 3e: Spatial uncertainty maps — range, sign agreement, and signal-to-noise

Three complementary diagnostics characterize *where* models agree and where they don't:

- **Range** (max − min): raw spread in absolute terms
- **Sign-agreement** (% of models agreeing on direction of change vs. baseline): the IPCC AR6 standard robustness criterion — regions below 80% agreement should not be used to draw conclusions about the direction of change
- **Signal-to-noise ratio** (MMM ÷ inter-model std): SNR > 1 means the forced signal exceeds model disagreement; SNR < 1 means the projection is not distinguishable from model noise

In [ ]:
# Compute cross-model statistics on the spatial fields at the warming level
mean_data = xy_by_warmlevel.tas.mean(dim="simulation")
std_data = xy_by_warmlevel.tas.std(dim="simulation")
min_data = xy_by_warmlevel.tas.min(dim="simulation")
max_data = xy_by_warmlevel.tas.max(dim="simulation")
range_data = max_data - min_data

# Sign-agreement: fraction of models projecting warming (anomaly > 0)
# xy_by_warmlevel is already an anomaly relative to 1850–1900, so > 0 means warmer
n_models = xy_by_warmlevel.sizes["simulation"]
n_positive = (xy_by_warmlevel.tas > 0).sum(dim="simulation")
# percent of models agreeing on warming
sign_agree = (n_positive / n_models) * 100

# Signal-to-noise ratio: MMM / inter-model std
# Clip std at a small floor to avoid division by zero in uniform regions
snr = mean_data / std_data.where(std_data > 0.01, other=0.01)

In [ ]:
cmap_div = read_ae_colormap(cmap="ae_diverging", cmap_hex=True)
cmap_blue = read_ae_colormap(cmap="ae_blue", cmap_hex=True)

range_vmax = np.nanpercentile(range_data.values, 99)
snr_vmax = np.nanpercentile(np.abs(snr.values), 99)


def make_map(data, title, vmin, vmax, cmap, width=200, height=200):
    return hv.QuadMesh((data["x"], data["y"], data)).opts(
        tools=["hover"],
        colorbar=True,
        cmap=cmap,
        clim=(vmin, vmax),
        xaxis=None,
        yaxis=None,
        clabel="Air Temperature (°C)",
        title=title,
        width=width,
        height=height,
    )

**Reading the maps:**
- **Range**: large values flag where individual model choice matters most for absolute outcomes.
- **Sign agreement**: regions below ~80% (lighter shading) are where models do not even agree on the *direction* of change — projections there should be interpreted with particular caution.
- **SNR > 1** (darker shading): the forced warming signal is distinguishable from model disagreement. SNR < 1 means the projection is dominated by inter-model noise rather than a coherent forced signal.

Coastal California typically has higher sign agreement and SNR than the interior, reflecting the moderating influence of the ocean on both the climate signal and model spread.

#### Step 3d: Regional dot plot — model spread by sub-region

Each dot is one model's temperature anomaly for a California sub-region at a chosen global warming level. The diamond marks the multi-model mean; the grey shadow spans the full model range. Use the dropdown to switch between warming levels. Hover for model names.

In [ ]:
# region definitions
REGIONS = {
    "All California": {"lat": (32.5, 42.0), "lon": (-124.5, -114.0)},
    "Coastal CA": {"lat": (34.0, 38.5), "lon": (-122.5, -119.5)},
    "Sierra Nevada": {"lat": (36.5, 41.0), "lon": (-120.5, -117.5)},
    "Central Valley": {"lat": (35.0, 40.0), "lon": (-122.0, -119.0)},
    "Southern CA": {"lat": (32.5, 35.0), "lon": (-120.5, -115.5)},
}

# pre-compute for all warming levels


def build_warmlevel_data(ds, wl):
    da_list = []
    for sim in ds.simulation:
        sim_name = sim.item()  # extract plain string from DataArray

        # gwl_times has a MultiIndex (simulation, ensemble, scenario)
        # build the correct tuple key matching the notebook's sim_idx logic
        if sim_name == "CESM2":
            key = (sim_name, "r11i1p1f1", "ssp370")
        elif sim_name == "CNRM-ESM2-1":
            key = (sim_name, "r1i1p1f2", "ssp370")
        else:
            key = (sim_name, "r1i1p1f1", "ssp370")

        if key not in gwl_times.index:
            continue

        year_str = str(gwl_times[str(wl)].loc[key])[:4]
        if len(year_str) == 4:
            da_list.append(ds.sel(time=int(year_str), simulation=sim_name))

    if not da_list:
        return None
    return xr.concat(da_list, dim="simulation")


xy_cae_anom = xy_anom.sel(simulation=cae_mdls_ls,
                          time=slice(hist_start, ssp_end))
print("Pre-computing warming level data...")
warmlevel_cache = {wl: build_warmlevel_data(
    xy_anom, wl) for wl in [1.5, 2.0, 2.5, 3.0]}
print("Done.")

In [ ]:
def region_mean(da, lat_range, lon_range):
    return da.sel(y=slice(*lat_range), x=slice(*lon_range)).mean(dim=["x", "y"])

In [ ]:
#  compute global x range across all warming levels and regions
all_vals = []
for wl_data in warmlevel_cache.values():
    if wl_data is None:
        continue
    for region_name, bounds in REGIONS.items():
        if region_name == "All California":
            continue
        for m in wl_data.simulation.values:
            v = float(
                region_mean(
                    wl_data.tas.sel(simulation=m), bounds["lat"], bounds["lon"]
                ).values
            )
            all_vals.append(v)

x_min = min(all_vals) - 0.2
x_max = max(all_vals) + 0.2

In [ ]:
REGION_COLORS = {
    "Coastal CA": "#4472A8",
    "Sierra Nevada": "#C0664A",
    "Central Valley": "#5A9E6F",
    "Southern CA": "#C8A83A",
}

#  plot function


def make_figure(wl):
    xy = warmlevel_cache[wl]
    if xy is None:
        print(f"No models reached {wl}°C.")
        return

    models = list(xy.simulation.values)
    n_regions = len([r for r in REGIONS if r != "All California"])

    fig = go.Figure()

    for ri, (region_name, bounds) in enumerate(
        [(r, b) for r, b in REGIONS.items() if r != "All California"]
    ):
        vals = [
            float(
                region_mean(
                    xy.tas.sel(simulation=m), bounds["lat"], bounds["lon"]
                ).values
            )
            for m in models
        ]
        mmm = float(np.mean(vals))
        jitter = np.linspace(-0.18, 0.18, len(models))

        # grey shadow spanning min to max
        fig.add_shape(
            type="rect",
            x0=min(vals),
            x1=max(vals),
            y0=ri - 0.35,
            y1=ri + 0.35,
            fillcolor="rgba(180,180,180,0.18)",
            line=dict(width=0),
            layer="below",
        )

        # individual model dots
        fig.add_trace(
            go.Scatter(
                x=vals,
                y=[ri + j for j in jitter],
                mode="markers",
                name=region_name,
                legendgroup=region_name,
                marker=dict(
                    color=REGION_COLORS[region_name],
                    size=10,
                    opacity=0.85,
                    line=dict(color="#000000", width=0.5),
                ),
                text=models,
                hovertemplate=(
                    "<b>%{text}</b><br>"
                    f"{region_name}<br>"
                    "Anomaly: +%{x:.2f}°C<extra></extra>"
                ),
            )
        )

        # MMM diamond
        fig.add_trace(
            go.Scatter(
                x=[mmm],
                y=[ri],
                mode="markers",
                name=f"{region_name} MMM",
                legendgroup=region_name,
                showlegend=False,
                marker=dict(
                    color="black",
                    size=8,
                    opacity=0.75,
                    symbol="diamond",
                ),
                hovertemplate=(
                    f"<b>{region_name} — multi-model mean</b><br>"
                    f"+{mmm:.2f}°C<extra></extra>"
                ),
            )
        )

    fig.update_layout(
        width=720,
        height=450,
        margin=dict(l=20, r=150, t=80, b=60),
        plot_bgcolor="white",
        paper_bgcolor="white",
        title=dict(
            text=(
                f"Model spread by region  ·  {wl}°C global warming (SSP3-7.0)<br>"
                "<sup>Dots = individual models · Diamond = multi-model mean · "
                "Shadow = full model range · Click legend to toggle</sup>"
            ),
            font=dict(size=13),
        ),
        xaxis=dict(
            title="Temperature anomaly (°C) relative to 1850–1900",
            range=[x_min, x_max],
            showgrid=True,
            gridcolor="#eeeeee",
            zeroline=False,
            showline=False,
            fixedrange=True,
        ),
        yaxis=dict(
            tickmode="array",
            tickvals=list(range(n_regions)),
            ticktext=[r for r in REGIONS if r != "All California"],
            showgrid=False,
            gridcolor="#eeeeee",
            range=[-0.4, n_regions - 0.4],
            showline=False,
            zeroline=False,
        ),
        legend=dict(
            orientation="v",
            yanchor="middle",
            y=0.5,
            xanchor="left",
            x=1.02,
            font=dict(size=10),
            itemclick="toggle",
            itemdoubleclick="toggleothers",
        ),
    )

    return fig


_ = interact(
    make_figure,
    wl=widgets.Dropdown(
        options=[1.5, 2.0, 2.5, 3.0],
        value=1.5,
        description="Warming (°C):",
        style={"description_width": "100px"},
    ),
)

In [ ]:
#  build one figure per warming level
wl_options = [1.5, 2.0, 2.5, 3.0]
all_traces = []
all_shapes = []
n_traces_per_wl = []
titles = []

for wl in wl_options:
    fig_wl = make_figure(wl)
    n_traces_per_wl.append(len(fig_wl.data))
    titles.append(
        f"Model spread by region  ·  {wl}°C global warming (SSP3-7.0)")
    all_traces.append(fig_wl.data)
    # all_shapes.append(list(fig_wl.layout.shapes))
    all_shapes.append([s.to_plotly_json() for s in fig_wl.layout.shapes])


#  combine into one figure
fig_combined = go.Figure()

for i, (traces, wl) in enumerate(zip(all_traces, wl_options)):
    for trace in traces:
        trace.visible = i == 0
        fig_combined.add_trace(trace)

#  dropdown visibility masks
buttons = []
idx = 0
for i, (wl, n) in enumerate(zip(wl_options, n_traces_per_wl)):
    visible = [False] * sum(n_traces_per_wl)
    for j in range(n):
        visible[idx + j] = True
    buttons.append(
        dict(
            label=f"{wl}°C",
            method="update",
            args=[
                {"visible": visible},
                {
                    "title.text": (
                        f"Model spread by region  ·  {wl}°C global warming (SSP3-7.0)<br>"
                        "<sup>Dots = individual models · Diamond = multi-model mean · "
                        "Shadow = full model range · Click legend to toggle</sup>"
                    ),
                    "shapes": all_shapes[i],
                },
            ],
        )
    )
    idx += n

# use layout from first figure as base
base_fig = make_figure(wl_options[0])
fig_combined.update_layout(base_fig.layout)

#  override with dropdown + annotation
fig_combined.update_layout(
    shapes=all_shapes[0],
    margin=dict(l=20, r=150, t=120, b=60),
    title=dict(
        text=(
            f"Model spread by region  ·  {wl_options[0]}°C global warming (SSP3-7.0)<br>"
            "<sup>Dots = individual models · Diamond = multi-model mean · "
            "Shadow = full model range · Click legend to toggle</sup>"
        ),
        font=dict(size=13),
        y=0.85,
        x=0.5,
        xanchor="center",
        yanchor="top",
    ),
    updatemenus=[
        dict(
            buttons=buttons,
            direction="down",
            showactive=True,
            x=0.22,
            xanchor="left",
            y=1.07,
            yanchor="top",
        )
    ],
)

fig_combined.add_annotation(
    text="Warming Level:",
    x=0.0,
    y=1.05,
    xref="paper",
    yref="paper",
    showarrow=False,
    font=dict(size=12),
    xanchor="left",
    yanchor="top",
)

fig_combined.show(config=config)
fig_combined.write_html(
    "figures/html/region_dots.html",
    include_plotlyjs="cdn",
    full_html=False,
    config=config,
)
fig_combined.write_image("figures/static/region_dots.png", scale=3)

### Step 4: When does model uncertainty matter most?

#### Step 4a: Warming-level scatter — global vs. regional response

Models that warm globally faster (reaching the warming level sooner) do not necessarily project proportionally more California warming. Each point is one model; the vertical spread at any x-position is pure model uncertainty in the California regional response.

In [ ]:
WARM_LEVELS = [1.5, 2.0, 3.0, 4.0]

# Combine all simulations for lookup
all_anom = xr.concat([cmip_anom, cae_anom], dim="simulation")


def build_scatter_df(wl):
    scenario = "ssp370"
    sim_idx_wl = []

    # Iterate over ALL simulations (cmip + cae)
    for simulation in all_anom.simulation.values:
        if simulation in gwl_times.index:
            if simulation == "CESM2":
                sim_idx_wl.append((simulation, "r11i1p1f1", scenario))
            elif simulation == "CNRM-ESM2-1":
                sim_idx_wl.append((simulation, "r1i1p1f2", scenario))
            else:
                sim_idx_wl.append((simulation, "r1i1p1f1", scenario))

    year_reached = []
    for i in sim_idx_wl:
        year_str = str(gwl_times[str(wl)].loc[i])[:4]
        if len(year_str) == 4:
            year_reached.append((i, int(year_str)))

    # Use combined dataset for end-of-century anomaly
    eoc_anom = all_anom.sel(time=slice(2071, 2100)).mean(dim="time")

    rows = []
    for sim_tuple, yr in year_reached:
        model = sim_tuple[0]
        try:
            ca_val = float(eoc_anom.sel(simulation=model).tas.values)
            # rows.append({"model": model, "year_wl": yr, "ca_anom": ca_val})
            rows.append(
                {
                    "model": model,
                    "year_wl": yr,
                    "ca_anom": ca_val,
                    "ca_anom_str": f"{ca_val:.2f}",  # pre-formatted string
                }
            )
        except Exception:
            pass

    return pd.DataFrame(rows)


print("Building scatter data for all warming levels...")
scatter_cache = {wl: build_scatter_df(wl) for wl in WARM_LEVELS}
print("Done.")

# Verify CAE models are now included
for wl, df in scatter_cache.items():
    cae_found = [m for m in cae_mdls_ls if m in df["model"].values]
    print(f"WL {wl}: {len(df)} models, CAE found: {cae_found}")

In [ ]:
WL_COLORS = {
    1.5: "#4472A8",
    2.0: "#5A9E6F",
    3.0: "#C8A83A",
    4.0: "#C0664A",
}

fig = go.Figure()

for wl, color in WL_COLORS.items():
    df = scatter_cache[wl]
    if df.empty:
        continue

    cae_df = df[df["model"].isin(cae_mdls_ls)]
    other_df = df[~df["model"].isin(cae_mdls_ls)]

    # Cal-Adapt AE — filled circle
    fig.add_trace(
        go.Scatter(
            x=cae_df["year_wl"],
            y=cae_df["ca_anom"],
            mode="markers",
            name=f"{wl}°C — Cal-Adapt AE",
            legendgroup=str(wl),
            marker=dict(
                color=color,
                size=10,
                symbol="circle",
                line=dict(color="white", width=1),
            ),
            text=cae_df["model"],
            hovertemplate=("<b>%{text}</b><br>" "Year reached: %{x}<br>"),
        )
    )

    # Other CMIP6 — hollow circle
    fig.add_trace(
        go.Scatter(
            x=other_df["year_wl"],
            y=other_df["ca_anom"],
            mode="markers",
            name=f"{wl}°C — Other CMIP6",
            legendgroup=str(wl),
            marker=dict(
                color="rgba(0,0,0,0)",  # transparent fill
                size=8,
                symbol="circle",
                line=dict(color=color, width=1.5),
            ),
            text=other_df["model"],
            hovertemplate=("<b>%{text}</b><br>" "Year reached: %{x}<br>"),
        )
    )

fig.update_layout(
    width=780,
    height=400,
    margin=dict(l=60, r=20, t=80, b=60),
    plot_bgcolor="white",
    paper_bgcolor="white",
    title=dict(
        text=(
            "Global warming rate vs. California end-of-century anomaly<br>"
            "<sup>by warming level — CMIP6 models (SSP3-7.0)</sup>"
        ),
        font=dict(size=13),
    ),
    xaxis=dict(
        title="Year model reaches warming level",
        showgrid=True,
        gridcolor="#eeeeee",
        zeroline=False,
        showline=False,
    ),
    yaxis=dict(
        title="California temperature anomaly<br>2071–2100 mean (°C)",
        showgrid=True,
        gridcolor="#eeeeee",
        zeroline=False,
        showline=False,
    ),
    legend=dict(
        orientation="v",
        yanchor="bottom",
        y=0,
        xanchor="left",
        x=1.02,
        font=dict(size=10),
        tracegroupgap=8,
        itemclick="toggle",
        itemdoubleclick="toggleothers",
    ),
)

fig.show(config=config)
fig.write_html(
    "../uncertainty/figures/html/wl_scatter.html",
    include_plotlyjs="cdn",
    full_html=False,
    config=config,
)
fig.write_image("../uncertainty/figures/static/wl_scatter.png", scale=2)

In [ ]:
all_anom = xr.concat([ssp_anom, cae_anom], dim="simulation")

In [ ]:
import plotly.graph_objects as go
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

np.random.seed(42)

# ── Color gradient by mean CA anomaly across all WLs ─────────────────────────
# Compute each model's mean ca_anom across all WLs for coloring
model_mean_anom = pd.concat(scatter_cache.values()).groupby("model")[
    "ca_anom"].mean()
vmin = model_mean_anom.min()
vmax = model_mean_anom.max()

# cold=blue, warm=red — swap to cmap_div if defined
# Build a matplotlib colormap from cmap_div
cmap = mcolors.LinearSegmentedColormap.from_list("cmap_div", cmap_div[1:-1])
plotly_colorscale = [
    [i / (len(cmap_div[1:-1]) - 1), c] for i, c in enumerate(cmap_div[1:-1])
]

model_mean_anom = pd.concat(scatter_cache.values()).groupby("model")[
    "ca_anom"].mean()
vmin = model_mean_anom.min()
vmax = model_mean_anom.max()
norm = mcolors.Normalize(vmin=vmin, vmax=vmax)


def model_color(model):
    val = model_mean_anom.get(model, (vmin + vmax) / 2)
    rgba = cmap(norm(val))
    return f"rgba({int(rgba[0]*255)},{int(rgba[1]*255)},{int(rgba[2]*255)},1)"


fig = go.Figure()
wls = sorted(scatter_cache.keys())
wl_to_y = {wl: i for i, wl in enumerate(wls)}
cae_set = set(m.strip() for m in cae_mdls_ls)

for wl in wls:
    df = scatter_cache[wl].copy()
    if df.empty:
        continue

    df["model"] = df["model"].astype(str)
    cae_df = df[df["model"].isin(cae_set)]
    other_df = df[~df["model"].isin(cae_set)]

    y_base = wl_to_y[wl]
    jitter = 0.12

    # Other CMIP6 — circles
    if len(other_df) > 0:
        fig.add_trace(
            go.Scatter(
                x=other_df["year_wl"],
                y=y_base + np.random.uniform(-jitter, jitter, len(other_df)),
                mode="markers",
                name="Other CMIP6",
                legendgroup="cmip6",
                showlegend=(wl == wls[0]),
                marker=dict(
                    symbol="circle",
                    color=[model_color(m) for m in other_df["model"]],
                    size=10,
                    line=dict(color="silver", width=1.0),
                ),
                customdata=list(
                    zip(
                        other_df["model"],
                        [wl] * len(other_df),
                        [
                            f"{v:.2f}" if not np.isnan(v) else "N/A"
                            for v in other_df["ca_anom"]
                        ],
                    )
                ),
                hovertemplate=(
                    "<b>%{customdata[0]}</b><br>"
                    "Warming level: %{customdata[1]}°C<br>"
                    "Year reached: %{x}<br>"
                    "CA anomaly: %{customdata[2]}°C"
                    "<extra></extra>"
                ),
            )
        )

    # Cal-Adapt AE — diamonds
    if len(cae_df) > 0:
        fig.add_trace(
            go.Scatter(
                x=cae_df["year_wl"],
                y=y_base + np.random.uniform(-jitter, jitter, len(cae_df)),
                mode="markers",
                name="Cal-Adapt AE",
                legendgroup="cae",
                showlegend=(wl == wls[0]),
                marker=dict(
                    symbol="diamond",
                    color=[model_color(m) for m in cae_df["model"]],
                    size=10,
                    line=dict(color="silver", width=1.0),
                ),
                customdata=list(
                    zip(
                        cae_df["model"],
                        [wl] * len(cae_df),
                        [
                            f"{v:.2f}" if not np.isnan(v) else "N/A"
                            for v in cae_df["ca_anom"]
                        ],
                    )
                ),
                hovertemplate=(
                    "<b>%{customdata[0]}</b><br>"
                    "Warming level: %{customdata[1]}°C<br>"
                    "Year reached: %{x}<br>"
                    "CA anomaly: %{customdata[2]}°C"
                    "<extra></extra>"
                ),
            )
        )

# Colorbar dummy trace
fig.add_trace(
    go.Scatter(
        x=[None],
        y=[None],
        mode="markers",
        marker=dict(
            colorscale=plotly_colorscale,
            cmin=vmin,
            cmax=vmax,
            color=[vmin, vmax],
            colorbar=dict(
                title=dict(
                    text="Mean CA<br>anomaly (°C)", side="right", font=dict(size=10)
                ),
                thickness=12,
                len=0.8,
                tickfont=dict(size=9),
            ),
            showscale=True,
            size=0,
        ),
        showlegend=False,
        hoverinfo="skip",
    )
)

fig.update_layout(
    width=700,
    height=400,
    margin=dict(l=80, r=120, t=80, b=60),
    plot_bgcolor="white",
    paper_bgcolor="white",
    title=dict(
        text=(
            "Global warming rate by model and warming level<br>"
            "<sup>Year each CMIP6 model reaches each warming level (SSP3-7.0)</sup>"
        ),
        font=dict(size=13),
    ),
    xaxis=dict(
        title="Year model reaches warming level",
        showgrid=True,
        gridcolor="#eeeeee",
        zeroline=False,
        showline=False,
    ),
    yaxis=dict(
        title="Global warming level",
        tickmode="array",
        tickvals=list(wl_to_y.values()),
        ticktext=[f"{wl}°C" for wl in wls],
        showgrid=True,
        gridcolor="#eeeeee",
        zeroline=False,
        showline=False,
        range=[-0.5, len(wls) - 0.5],
    ),
    legend=dict(
        orientation="v",
        yanchor="bottom",
        y=0.02,
        xanchor="left",
        x=1.12,
        font=dict(size=10),
    ),
)

fig.show(config=config)
fig.write_html(
    "figures/html/wl_scatter.html",
    include_plotlyjs="cdn",
    full_html=False,
    config=config,
)
fig.write_image("figures/static/wl_scatter.png", scale=2)

### Step 5: Inter-model spread across variables

Model uncertainty is not the same for all variables. Temperature projections have relatively low inter-model spread (high SNR), while precipitation is far more uncertain — models often disagree on both the magnitude *and* the sign of future changes. This section makes that contrast concrete using the same CMIP6 archive.

#### Step 5a: Retrieve precipitation data

In [ ]:
# Retrieve CMIP6 precipitation (pr) — same spatial domain and models as temperature
copt_pr = CmipOpt()
copt_pr.variable = "pr"  # precipitation
copt_pr.area_subset = "states"
copt_pr.location = "California"
copt_pr.area_average = False
copt_pr.timescale = "Amon"

mdls_pr = grab_multimodel_data(copt_pr)

In [ ]:
# Annual average → convert to mm/day → anomaly relative to 1981–2010 climatology
pr_yr = weighted_temporal_mean(mdls_pr).compute()
pr_yr = pr_yr * 86400  # kg m-2 s-1 → mm day-1
pr_yr.pr.attrs["units"] = "mm/day"

# Use a more recent baseline for precipitation (1981–2010 is the WMO standard)
pr_anom = calc_anom(pr_yr, base_start=1981, base_end=2010)
pr_anom.pr.attrs["units"] = "mm/day anomaly"

#### Step 5b: Coefficient of variation — normalised spread comparison

The **coefficient of variation** (CV = std / |mean|) normalises inter-model spread by the signal magnitude, making it comparable across variables with different units and scales. Higher CV = more model disagreement relative to the projected change.

In [ ]:
#  compute CV (std / |mean|) for each variable over end-of-century
# requires: xy_anom (temperature) and pr_anom (precipitation) from the notebook
# both already computed in Step 5

eoc_slice = slice(2071, 2100)


def compute_cv(da, dim="simulation"):
    """Coefficient of variation: inter-model std / |inter-model mean|."""
    mean = da.sel(time=eoc_slice).mean(dim=["time", dim])
    std = da.sel(time=eoc_slice).std(dim=dim).mean(dim="time")
    # floor the mean to avoid division near zero
    return float((std / mean.where(np.abs(mean) > 0.05, other=0.05)).mean())


cv_tas = compute_cv(xy_anom.tas)
cv_pr = compute_cv(pr_anom.pr)

#  if you have wind speed loaded, add it here
# cv_wind = compute_cv(wind_anom.sfcWind)

variables = ["Max temperature", "Precipitation"]
cv_values = [cv_tas, cv_pr]
colors = ["#4472A8", "#C0664A"]

#  plot
fig, ax = plt.subplots(figsize=(6, 2.5))

bars = ax.barh(
    variables,
    cv_values,
    color=colors,
    height=0.45,
    edgecolor="white",
    linewidth=0.5,
)

# value labels
for bar, val in zip(bars, cv_values):
    ax.text(
        val + 0.003,
        bar.get_y() + bar.get_height() / 2,
        f"{val:.2f}",
        va="center",
        ha="left",
        fontsize=10,
        color="#444",
    )

# threshold line at CV = 1 (spread equals signal)
ax.axvline(1.0, color="#888", lw=1, ls="dashed", alpha=0.6)
ax.text(
    1.02,
    0.02,
    "CV = 1\n(spread = signal)",
    transform=ax.get_yaxis_transform(),
    fontsize=8,
    color="#888",
    va="bottom",
)

ax.set_xlabel(
    "Coefficient of variation (inter-model std / |mean|)", fontsize=10)
ax.set_title(
    "Inter-model spread by variable — end of century, SSP3-7.0\n"
    "Higher CV = more model disagreement relative to projected change",
    fontsize=10,
    pad=10,
)
ax.spines[["top", "right"]].set_visible(False)
ax.tick_params(labelsize=10)
ax.set_xlim(left=0)
ax.grid(axis="x", alpha=0.2, lw=0.6)

plt.tight_layout()

fig.savefig("figures/static/variable_cv.png", dpi=150)

Precipitation CV is substantially higher than temperature CV across most of California — meaning that the choice of model matters far more for precipitation-dependent applications (water supply, flood risk) than for heat-related ones. Regions where the precipitation anomaly mean is near zero (models split between wetting and drying) produce very high CV values, which is itself a signal of deep model disagreement.

# IPCC AR6 Interactive Atlas — CMIP6 Model Uncertainty

Explores CMIP6 multi-model spread across climate variables for **Western North America** under **SSP3-7.0**, using percentile data extracted from the [IPCC WGI Interactive Atlas](https://interactive-atlas.ipcc.ch/regional-information).

Projected changes are expressed relative to the **1850–1900 baseline** and shown for three future time periods using box-whisker style plots. Three line widths encode different percentile ranges (P5–P95, P10–P90, P25–P75) with the median marked as a dot.

---

## 1. Imports

In [ ]:
import pandas as pd
import plotly.graph_objects as go

## 2. Data

Raw percentile statistics (P5, P10, P25, median, P75, P90, P95) were manually extracted from the [IPCC WGI Interactive Atlas: Regional information](https://interactive-atlas.ipcc.ch/regional-information) for five climate variables across three future time periods (**Near Term 2021–2040**, **Medium Term 2041–2060**, **Long Term 2081–2100**) under **SSP3-7.0** for **Western North America**.

In [ ]:
# Raw data from IPCC AR6 Interactive Atlas — Western North America, SSP3-7.0
data = {
    "Mean temperature (°C)": {
        "Near Term (2021–2040)":   {"median": 1.9, "p25": 1.6, "p75": 2.2, "p10": 1.3, "p90": 2.4, "p5": 1.1,  "p95": 2.7},
        "Medium Term (2041–2060)": {"median": 2.7, "p25": 2.4, "p75": 3.3, "p10": 2.1, "p90": 3.6, "p5": 1.9,  "p95": 4.0},
        "Long Term (2081–2100)":   {"median": 4.8, "p25": 4.2, "p75": 5.4, "p10": 3.8, "p90": 6.1, "p5": None, "p95": None},
    },
    "Max temperature (°C)": {
        "Near Term (2021–2040)":   {"median": 1.9, "p25": 1.6, "p75": 2.1, "p10": 1.6, "p90": 2.3, "p5": 1.2,  "p95": 2.7},
        "Medium Term (2041–2060)": {"median": 2.7, "p25": 2.4, "p75": 3.0, "p10": 2.2, "p90": 3.7, "p5": 2.0,  "p95": 4.0},
        "Long Term (2081–2100)":   {"median": 4.8, "p25": 4.2, "p75": 5.5, "p10": 4.1, "p90": 6.4, "p5": 3.8,  "p95": 6.6},
    },
    "Total precipitation (%)": {
        "Near Term (2021–2040)":   {"median": 1.9,  "p25": -0.9, "p75": 3.7, "p10": -2.1, "p90": 5.8,  "p5": -2.7, "p95": 8.9},
        "Medium Term (2041–2060)": {"median": 3.3,  "p25":  1.4, "p75": 4.6, "p10": -0.1, "p90": 8.8,  "p5": -1.1, "p95": 10.6},
        "Long Term (2081–2100)":   {"median": 7.7,  "p25":  3.7, "p75": 9.6, "p10":  2.1, "p90": 13.3, "p5": -0.7, "p95": 13.8},
    },
    "Surface wind (%)": {
        "Near Term (2021–2040)":   {"median": -1.0, "p25": -1.7, "p75": -0.0, "p10": -2.6, "p90": 1.2,  "p5": -4.0, "p95": 2.7},
        "Medium Term (2041–2060)": {"median": -1.6, "p25": -2.2, "p75": -0.8, "p10": -4.2, "p90": 0.3,  "p5": -4.4, "p95": 2.2},
        "Long Term (2081–2100)":   {"median": -3.2, "p25": -4.6, "p75": -2.3, "p10": -8.4, "p90": -1.1, "p5": -8.9, "p95": 1.0},
    },
    "SST (°C)": {
        "Near Term (2021–2040)":   {"median": 1.2, "p25": 0.8, "p75": 1.4, "p10": 0.4, "p90": 1.6, "p5": 0.3, "p95": 1.7},
        "Medium Term (2041–2060)": {"median": 1.8, "p25": 1.4, "p75": 2.0, "p10": 1.1, "p90": 2.4, "p5": 1.0, "p95": 2.8},
        "Long Term (2081–2100)":   {"median": 3.1, "p25": 2.7, "p75": 3.7, "p10": 2.3, "p90": 4.2, "p5": 2.2, "p95": 5.0},
    },
}

In [ ]:
# Build dataframe
rows = []
for var, periods in data.items():
    for period, vals in periods.items():
        rows.append({"variable": var, "period": period, **vals})

df = pd.DataFrame(rows)

# derived spread metrics
df["iqr"] = df["p75"] - df["p25"]          # interquartile range
df["range_80"] = df["p90"] - df["p10"]          # 10th–90th range
df["cv"] = df["iqr"] / df["median"].abs()  # coefficient of variation

print(df[["variable", "period", "median", "p25", "p75",
          "iqr", "range_80", "cv"]].to_string(index=False))

## 3. Visualization

Box-whisker plot of CMIP6 model spread per variable and time period. Each variable shows three jittered rows — one per period — using color to distinguish them. The chart is exported as both interactive HTML and a high-resolution PNG.

In [ ]:
# Interactive plot
periods = ["Near Term (2021–2040)",
           "Medium Term (2041–2060)", "Long Term (2081–2100)"]
variables = list(data.keys())

COLORS = {
    "Near Term (2021–2040)":   "#4472A8",
    "Medium Term (2041–2060)": "#C0664A",
    "Long Term (2081–2100)":   "#5A9E6F",
}

PERIOD_SHORT = {
    "Near Term (2021–2040)":   "Near Term",
    "Medium Term (2041–2060)": "Medium Term",
    "Long Term (2081–2100)":   "Long Term",
}

fig = go.Figure()

for vi, var in enumerate(variables):
    for pi, period in enumerate(periods):
        row = df[(df["variable"] == var) & (df["period"] == period)].iloc[0]
        color = COLORS[period]
        offset = (pi - 1) * 0.22   # jitter periods within each variable

        # p5–p95 whisker (thin)
        if row["p5"] is not None and row["p95"] is not None:
            fig.add_trace(go.Scatter(
                x=[row["p5"], row["p95"]],
                y=[vi + offset, vi + offset],
                mode="lines",
                line=dict(color=color, width=1.5),
                showlegend=False,
            ))

        # p10–p90 bar (medium)
        fig.add_trace(go.Scatter(
            x=[row["p10"], row["p90"]],
            y=[vi + offset, vi + offset],
            mode="lines",
            line=dict(color=color, width=4),
            showlegend=False,
        ))

        # p25–p75 bar (thick)
        fig.add_trace(go.Scatter(
            x=[row["p25"], row["p75"]],
            y=[vi + offset, vi + offset],
            mode="lines",
            line=dict(color=color, width=9),
            showlegend=False,
        ))

        # median dot
        fig.add_trace(go.Scatter(
            x=[row["median"]],
            y=[vi + offset],
            mode="markers",
            marker=dict(color="white", size=8,
                        line=dict(color=color, width=2)),
            name=PERIOD_SHORT[period],
            legendgroup=period,
            showlegend=(vi == 0),   # only show legend once per period
        ))

        # zero reference line
        fig.add_vline(x=0, line=dict(color="grey", width=0.8, dash="dash"))

fig.update_layout(
    title=dict(
        text=(
            "CMIP6 model uncertainty by variable — Western North America (SSP3-7.0)<br>"
            "<sup>Source: IPCC AR6 Interactive Atlas · "
            "Thick bar = P25–P75 · Medium bar = P10–P90 · Thin line = P5–P95 · Dot = median</sup>"
        ),
        font=dict(size=13),
        x=0.5,
        xanchor="center",
    ),
    xaxis_title="Projected change relative to 1850–1900",
    height=480,
    width=700,
    plot_bgcolor="white",
    margin=dict(l=200, r=30, t=100, b=60),
    legend=dict(
        orientation="h",
        y=-0.15,
        x=0.5,
        xanchor="center",
        font=dict(size=11),
    ),
    yaxis=dict(
        tickmode="array",
        tickvals=list(range(len(variables))),
        ticktext=variables,
        showgrid=False,
        zeroline=False,
    ),
    xaxis=dict(
        showgrid=True,
        gridcolor="#eeeeee",
        zeroline=False,
    ),
)

fig.show(config=config)
fig.write_html(
    "figures/html/ipcc_atlas_uncertainty.html",
    include_plotlyjs="cdn",
    full_html=False,
    config=config,
)
fig.write_image("figures/static/ipcc_atlas_uncertainty.png", scale=3)